In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.utils as vutils
from torch.utils.data import DataLoader
import os
import pennylane as qml
from pennylane import numpy as npq
import matplotlib.pyplot as plt
from torchvision import models
from torch.nn.utils import spectral_norm
import os
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import random
import numpy as np
from tqdm.auto import tqdm

/home/students/miniconda3/envs/asma/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -----------------------------------------------------------------
# Set this flag to True to use the QuantumBlock, or False to use a
# classical alternative in the Generator Network.
# -----------------------------------------------------------------
USE_QUANTUM = False

In [3]:
# -----------------------------------------------------------------
# Quantum Circuit & Block 
# -----------------------------------------------------------------
n_qubits = 5
dev = qml.device("lightning.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    inputs = inputs.to(torch.float32)
    weights = weights.to(torch.float32)

    # Encode each input into a qubit rotation (RY gate)
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)

    # Entangle with CNOT gates
    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i + 1])

    # Parameterized rotations on each qubit
    weights_reshaped = weights.view(n_qubits, 3)
    for i in range(n_qubits):
        qml.RX(weights_reshaped[i, 0], wires=i)
        qml.RY(weights_reshaped[i, 1], wires=i)
        qml.RZ(weights_reshaped[i, 2], wires=i)

    # Return expectation values on each qubit
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

class QuantumBlock(nn.Module):
    """
    Wraps the above quantum circuit into a PyTorch module.
    """
    def __init__(self, n_qubits=5):
        super(QuantumBlock, self).__init__()
        # We have n_qubits * 3 trainable parameters
        self.weights = nn.Parameter(
            torch.randn(n_qubits * 3, dtype=torch.float32) * 0.01
        )
        self.n_qubits = n_qubits

    def forward(self, x):
        """
        x: shape (batch_size, n_qubits)
        Returns: shape (batch_size, n_qubits) => each row is [Z_0, ..., Z_(n_qubits-1)]
        """
        x = x.to(torch.float32)
        out_list = []
        for i in range(x.shape[0]):
            q_out = quantum_circuit(x[i], self.weights)
            # q_out is a list/tensor of length n_qubits
            q_out = torch.stack(q_out)  # shape (n_qubits,)
            out_list.append(q_out.to(torch.float32))
        return torch.stack(out_list, dim=0)  # shape (batch_size, n_qubits)


class ClassicalBlock(nn.Module):
    """
    A classical alternative to the quantum block with ~18 parameters 
    (if n_qubits=6). Just a placeholder to match parameter counts.
    """
    def __init__(self, n_qubits):
        super(ClassicalBlock, self).__init__()
        # no bias in the first layer
        self.linear1 = nn.Linear(n_qubits, 1, bias=False)
        self.linear2 = nn.Linear(1, n_qubits, bias=True)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # x: (batch_size, n_qubits)
        x = self.linear1(x)       # => (batch_size, 1)
        x = self.relu(x)
        x = self.linear2(x)       # => (batch_size, n_qubits)
        return x

In [4]:
# -----------------------------------------------------------------
# Residual Upsampling Block (used in the generator)
# -----------------------------------------------------------------
class ResBlockUp(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResBlockUp, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.up = nn.Upsample(scale_factor=2, mode='nearest')
        self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        x_res = self.up(x)
        out = self.bn1(self.conv1(x_res))
        out = self.relu(out)
        out = self.bn2(self.conv2(out))
        sc = self.shortcut(x_res)
        out += sc
        out = self.relu(out)
        return out

In [5]:
# -----------------------------------------------------------------
# Generator: Uses either a QuantumBlock or a ClassicalBlock
# -----------------------------------------------------------------
class Generator(nn.Module):
    def __init__(self, latent_dim=4, ngf=64, n_qubits=4, use_quantum=True):
        super(Generator, self).__init__()
        self.latent_dim = latent_dim
        self.n_qubits = n_qubits
        self.use_quantum = use_quantum
        print("latent",self.latent_dim )
        print("qubits",self.n_qubits )
        print("Quantum in G?",self.use_quantum)

        # The first n_qubits elements of the latent vector are processed
        if use_quantum:
            self.qblock = QuantumBlock(n_qubits=self.n_qubits)
        else:
            self.cblock = ClassicalBlock(self.n_qubits)
        
        # Map the 32-dimensional output from the block to a feature map
        self.fc = nn.Linear(self.n_qubits, ngf * 4 * 4 * 4)
        self.bn0 = nn.BatchNorm1d(ngf * 4 * 4 * 4)
        self.relu = nn.ReLU(True)
        # Two residual upsampling blocks to gradually increase spatial dimensions
        self.resup1 = ResBlockUp(ngf * 4, ngf * 2)
        self.resup2 = ResBlockUp(ngf * 2, ngf)
        self.upsample_final = nn.Upsample(size=32)
        # Final convolution to get a 3-channel output (CIFAR-10)
        self.conv_final = nn.Conv2d(ngf, 3, kernel_size=3, stride=1, padding=1)
        self.tanh = nn.Tanh()

    def forward(self, z):
        # Extract only the first n_qubits dimensions from the latent vector
        if self.use_quantum:
            quantum_input = z#[:, :self.n_qubits]
            block_out = self.qblock(quantum_input)
        else:
            classical_input = z#[:, :self.n_qubits]
            block_out = self.cblock(classical_input)
        out = self.fc(block_out)
        out = self.bn0(out)
        out = self.relu(out)
        out = out.view(-1, 256, 4, 4)  # Reshape into feature maps
        out = self.resup1(out)
        out = self.resup2(out)
        out = self.upsample_final(out)
        out = self.conv_final(out)
        img = self.tanh(out)
        return img

In [6]:
##############################################################################
# HYBRID DISCRIMINATOR WITH A QUANTUM LAST LAYER
##############################################################################
class Discriminator(nn.Module):
    """
    A  ResNet18-based critic that uses a single spectral_norm on conv1, removing maxpool, etc.
    and retains a QuantumBlock at the end for the final classification.
    """
    def __init__(self, n_qubits=5):
        super(Discriminator, self).__init__()
        self.n_qubits = n_qubits

        # 1) Create a smaller ResNet18 with spectral_norm on conv1
        self.model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.model.conv1 = spectral_norm(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        )
        # Remove maxpool to preserve more spatial info
        self.model.maxpool = nn.Identity() ###############!!!!!!!!!!!!!!! 
        
        # 2) Instead of having a final (num_ftrs->1) layer, we do Identity
        #num_ftrs = self.model.fc.in_features

        self.model.fc = nn.Identity()

        # 3) Map the extracted features to n_qubits => feed to QuantumBlock
        self.fc_to_q_input = spectral_norm(nn.Linear(512, n_qubits))
        
        # 4) The quantum block: processes an (N, n_qubits) => (N, n_qubits)
        
        self.qblock_d = QuantumBlock(n_qubits=self.n_qubits)

        # 5) Final fully connected to produce a single real score
        self.fc_out = nn.Linear(n_qubits, 1)
        
        for param in self.model.parameters():
            param.requires_grad = True
   
    def forward(self, x):
        # Preprocess for ResNet
        x = F.interpolate(x, size=(224,224), mode='bilinear', align_corners=False)
        
        # Pass through ResNet (conv1..layer4, then avgpool) => feature vector
        features = self.model(x)  # shape: (batch_size, num_ftrs)

        # Convert features => (batch_size, n_qubits)
        q_input = self.fc_to_q_input(features)

        # Pass into quantum circuit => (batch_size, n_qubits)
        q_out = self.qblock_d(q_input)
        q_out = q_out.to(device)

        # Single scalar score => (batch_size,)
        logits = self.fc_out(q_out).squeeze(-1)
        return logits

In [7]:
# -----------------------------------------------------------------
# Data Preparation: CIFAR-10
# -----------------------------------------------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])
train_dataset = datasets.CIFAR10(root='data', train=True, download=True, transform=transform)

# Set target_class to 2 (bird)
target_class = 2

# Filter the dataset to include only images with label 2
indices = [i for i, label in enumerate(train_dataset.targets) if label == target_class]

# Create a subset containing only the bird images
subset_train_dataset = Subset(train_dataset, indices)

#dataloader = DataLoader(subset_train_dataset, batch_size=8, shuffle=True, num_workers=2)

def make_dataloader(dataset, batch_size, seed):
    g = torch.Generator()
    g.manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        generator=g,
        drop_last=False
    )

Files already downloaded and verified


In [8]:
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Determinism (slower but reproducible)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
import os
import torchvision.utils as vutils

def save_fake_images(G, seed, latent_dim, device, out_dir, n_images=10000, batch_size=8):
    os.makedirs(out_dir, exist_ok=True)
    G.eval()
    idx = 0
    with torch.no_grad():
        while idx < n_images:
            b = min(batch_size, n_images - idx)
            z = torch.randn(b, latent_dim, device=device)
            fake = G(z)                       # in [-1,1]
            fake = (fake + 1) / 2             # to [0,1]
            for j in range(b):
                vutils.save_image(fake[j], os.path.join(out_dir, f"{idx+j:06d}.png"))
            idx += b
    G.train()

In [ ]:
def train_one_seed(seed, train_dataset, num_epochs=100, batch_size=8, device=None):
    seed_everything(seed)

    # Rebuild dataloader with this seed
    dataloader = make_dataloader(train_dataset, batch_size=batch_size, seed=seed)

    # Reinitialize models/optimizers fresh EACH seed
    latent_dim = n_qubits
    G = Generator(latent_dim=latent_dim, n_qubits=n_qubits, use_quantum=USE_QUANTUM).to(device)
    D = Discriminator().to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizerG = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    optimizerD = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

    fixed_noise = torch.randn(8, latent_dim, device=device)

    d_loss_per_epoch = []
    g_loss_per_epoch = []

    for epoch in range(num_epochs):
        d_running = 0.0
        g_running = 0.0
        n_batches = 0
        pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
        for i, (imgs, _) in pbar:
            imgs = imgs.to(device)
            b_size = imgs.size(0)

            # ---- Train D ----
            D.zero_grad()
            real_labels = torch.ones(b_size, device=device)
            fake_labels = torch.zeros(b_size, device=device)

            real_pred = D(imgs)
            d_loss_real = criterion(real_pred, real_labels)

            z = torch.randn(b_size, latent_dim, device=device)
            fake_imgs = G(z)

            fake_pred = D(fake_imgs.detach())
            d_loss_fake = criterion(fake_pred, fake_labels)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            optimizerD.step()

            # ---- Train G ----
            G.zero_grad()
            gen_pred = D(fake_imgs)
            g_loss = criterion(gen_pred, real_labels)
            g_loss.backward()
            optimizerG.step()

            # accumulate
            d_running += d_loss.item()
            g_running += g_loss.item()
            n_batches += 1
            pbar.set_postfix(D=f"{d_loss.item():.4f}",
                             G=f"{g_loss.item():.4f}")
        # epoch mean
        d_loss_per_epoch.append(d_running / n_batches)
        g_loss_per_epoch.append(g_running / n_batches)
    
    torch.save(
        {"seed": seed, "G_state": G.state_dict(), "config": {"n_qubits": n_qubits, "latent_dim": latent_dim}},
        f"Exp2/G_seed{seed}.pth"
    )
    # Generate To compute FID/KID/IS Later
    save_fake_images(G, seed, latent_dim, device, out_dir=f"Exp2/eval_fakes/seed_{seed}", n_images=1000)

    return np.array(d_loss_per_epoch), np.array(g_loss_per_epoch)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
if device.type == "cuda":
    torch.cuda.set_device(0)
seeds = [7, 17, 27]  
num_epochs = 100
all_d = []
all_g = []

for s in seeds:
    print("Seed: ", s)
    d_ep, g_ep = train_one_seed(
        seed=s,
        train_dataset=subset_train_dataset,
        num_epochs=num_epochs,
        batch_size=8,
        device=device
    )
    np.savez(f"Exp2/loss_seed_{s}.npz",d_ep=d_ep,g_ep=g_ep,seed=s)
    all_d.append(d_ep)
    all_g.append(g_ep)

all_d = np.stack(all_d, axis=0)  # (n_seeds, n_epochs)
all_g = np.stack(all_g, axis=0)

np.savez(f"Exp2/losses_across_seeds.npz",all_d=all_d,all_g=all_g,seeds=np.array(seeds))

d_mean, d_std = all_d.mean(axis=0), all_d.std(axis=0, ddof=1)
g_mean, g_std = all_g.mean(axis=0), all_g.std(axis=0, ddof=1)

In [ ]:
epochs = np.arange(1, num_epochs + 1)

plt.figure(figsize=(8,5))
plt.plot(epochs, d_mean, label="D loss (mean)")
plt.fill_between(epochs, d_mean - d_std, d_mean + d_std, alpha=0.2, label="D loss ± std")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Discriminator Loss Across Seeds (mean ± std)")
plt.legend()
plt.tight_layout()
plt.savefig(f"Exp2/ex2_D_loss_mean_std_seeds.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(epochs, g_mean, label="G loss (mean)")
plt.fill_between(epochs, g_mean - g_std, g_mean + g_std, alpha=0.2, label="G loss ± std")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Generator Loss Across Seeds (mean ± std)")
plt.legend()
plt.tight_layout()

plt.savefig(f"Exp2/ex2_G_loss_mean_std_seeds.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(epochs, d_mean, label="Discriminator mean")
plt.fill_between(epochs, d_mean - d_std, d_mean + d_std, alpha=0.2, label="Discriminator ± std")

plt.plot(epochs, g_mean, label="Generator mean")
plt.fill_between(epochs, g_mean - g_std, g_mean + g_std, alpha=0.2, label="Generator ± std")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Curves Across 3 Seeds")
plt.legend()
plt.tight_layout()
plt.savefig(f"Exp2/loss_mean_std_combined.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def save_real_images(dataset, out_dir, n_images=None):
    os.makedirs(out_dir, exist_ok=True)
    n = len(dataset) if n_images is None else min(n_images, len(dataset))
    for i in range(n):
        x, _ = dataset[i]              # tensor normalized already in your dataset
        x = (x + 1) / 2                # back to [0,1]
        vutils.save_image(x, os.path.join(out_dir, f"{i:06d}.png"))
#Prepare Reals for KID/FID/IS
test_dataset = datasets.CIFAR10(
    root='data',
    train=False,
    download=True,
    transform=transform
)
indices_test = [i for i, label in enumerate(test_dataset.targets) if label == 2]
subset_test_dataset = Subset(test_dataset, indices_test)

save_real_images(subset_test_dataset, out_dir="eval_reals", n_images=1000)

In [4]:
from torch_fidelity import calculate_metrics

def compute_metrics(fake_dir,real_dir, cuda=True):
    metrics = calculate_metrics(
        input1=fake_dir,
        input2=real_dir,
        cuda=cuda,
        fid=True,
        kid=True,
        isc=True,      # inception score
        verbose=False,
        prc=True, 
        dataset_name="cifar10"
    )
    return metrics

In [5]:
seeds = [7, 17, 27]
all_metrics = []

for s in seeds:
    m = compute_metrics(f"Exp2/eval_fakes/seed_{s}", "eval_reals", cuda=torch.cuda.is_available())
    all_metrics.append(m)
    print(s, m)

7 {'inception_score_mean': 2.1685135676667207, 'inception_score_std': 0.14301004859340877, 'frechet_inception_distance': 337.18451758003596, 'kernel_inception_distance_mean': 0.24370009308058083, 'kernel_inception_distance_std': 1.2407695278665974e-07, 'precision': 0.8259999752044678, 'recall': 0.0, 'f_score': 0.0}
17 {'inception_score_mean': 2.3765951080205605, 'inception_score_std': 0.15653483948031488, 'frechet_inception_distance': 316.38487934055576, 'kernel_inception_distance_mean': 0.2308535210735736, 'kernel_inception_distance_std': 1.018270593767553e-07, 'precision': 0.8560000061988831, 'recall': 0.0, 'f_score': 0.0}
27 {'inception_score_mean': 2.151726283925888, 'inception_score_std': 0.06564069394246849, 'frechet_inception_distance': 295.82719180631545, 'kernel_inception_distance_mean': 0.21065427103353357, 'kernel_inception_distance_std': 1.677629461856084e-07, 'precision': 0.8999999761581421, 'recall': 0.0, 'f_score': 0.0}
